# 5060 Final Project — Literature Search Pipeline

**Topic:** SNOMED-CT in Clinical NLP — A Scoping Review (2016–2026)

**Sections:**
0. Environment Setup
1. Clinical NLP Task Vocabulary
2. Database Registry
3. Stage 1 Metadata Schema
4. Query Construction
5. Paper Retrieval
6. Deduplication
7. Screening

---
## 0. Environment Setup

Run this cell **once** from any kernel to create the `meta_review` conda environment
and register it as a Jupyter kernel. Then restart Jupyter and select
**Python (meta_review)** as the kernel before running the rest of the notebook.

In [ ]:
%%bash
conda create -n meta_review python=3.10 -y

conda run -n meta_review pip install \
    biopython \
    arxiv \
    requests \
    acl-anthology \
    bibtexparser \
    pandas \
    "thefuzz[speedup]" \
    python-dotenv \
    tqdm \
    ipykernel

conda run -n meta_review python -m ipykernel install \
    --user \
    --name meta_review \
    --display-name "Python (meta_review)"

echo "Done. Restart Jupyter and select the 'Python (meta_review)' kernel."

---
## Imports

In [ ]:
import json
import os
import time
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import requests
import pandas as pd
import bibtexparser
import arxiv
from Bio import Entrez, Medline
from dotenv import load_dotenv
from thefuzz import fuzz
from tqdm import tqdm

---
## 1. Clinical NLP Task Vocabulary

In [ ]:
VOCABULARY = {
    "meta": {
        "version": "1.0",
        "date": "2026-04-27",
        "anchor_papers": [
            {"id": "gao2022",      "citation": "Gao Y et al. JAMIA. 2022;29(10):1797-1806. doi:10.1093/jamia/ocac127"},
            {"id": "wu2022",       "citation": "Wu H et al. npj Digit Med. 2022;5:186. doi:10.1038/s41746-022-00730-6"},
            {"id": "lederman2022", "citation": "Lederman A et al. JAMIA. 2022;29(10):1810-1817. doi:10.1093/jamia/ocac121"}
        ]
    },
    "categories": [
        {"id": 1,  "name": "Named Entity Recognition",                          "search_terms": ["named entity recognition", "concept extraction", "entity extraction", "clinical NER", "disorder detection", "medical event identification"]},
        {"id": 2,  "name": "De-identification",                                 "search_terms": ["de-identification", "deidentification", "PHI detection", "anonymization", "patient identifier"]},
        {"id": 3,  "name": "Entity Linking / Concept Normalization",             "search_terms": ["entity linking", "concept normalization", "entity normalization", "concept mapping", "terminology mapping", "clinical concept normalization"]},
        {"id": 4,  "name": "Relation Extraction",                               "search_terms": ["relation extraction", "clinical relation extraction", "event extraction"]},
        {"id": 5,  "name": "Adverse Drug Event / Medication Extraction",        "search_terms": ["adverse drug event", "adverse drug reaction", "medication extraction", "drug extraction", "pharmacovigilance"]},
        {"id": 6,  "name": "Text / Document Classification",                   "search_terms": ["text classification", "document classification", "clinical phenotyping", "cohort selection", "smoking status", "risk assessment"]},
        {"id": 7,  "name": "Assertion / Negation Detection",                   "search_terms": ["assertion detection", "negation detection", "uncertainty detection", "hedge detection"]},
        {"id": 8,  "name": "Summarization",                                    "search_terms": ["summarization", "text summarization", "clinical summarization", "discharge summary", "radiology report summarization"]},
        {"id": 9,  "name": "Question Answering / MRC",                         "search_terms": ["question answering", "machine reading comprehension", "clinical question answering", "medical question answering"]},
        {"id": 10, "name": "Information Retrieval",                            "search_terms": ["information retrieval", "clinical information retrieval"]},
        {"id": 11, "name": "Temporal Expression Extraction",                   "search_terms": ["temporal information extraction", "temporal expression", "temporal relation", "clinical timeline"]},
        {"id": 12, "name": "Coreference Resolution",                           "search_terms": ["coreference resolution", "co-reference resolution"]},
        {"id": 13, "name": "Semantic Textual Similarity / NLI",                "search_terms": ["semantic textual similarity", "natural language inference", "textual entailment", "clinical NLI"]},
        {"id": 14, "name": "Natural Language Generation",                      "search_terms": ["natural language generation", "text generation", "clinical text generation"]},
        {"id": 15, "name": "Clinical Prediction / Outcome",                   "search_terms": ["readmission prediction", "outcome prediction", "prognosis", "length of stay", "clinical prediction"]},
        {"id": 16, "name": "Quality / Workflow",                               "search_terms": ["clinical audit", "workflow", "procedure quality"]}
    ]
}

ALL_TASK_TERMS = [t for cat in VOCABULARY["categories"] for t in cat["search_terms"]]
print(f"{len(VOCABULARY['categories'])} categories, {len(ALL_TASK_TERMS)} total search terms")

In [ ]:
Path("vocabulary.json").write_text(json.dumps(VOCABULARY, indent=2))
print("Saved vocabulary.json")

---
## 2. Database Registry

In [ ]:
DATABASES = {
    "tier1_primary": [
        {"id": "pubmed",        "name": "PubMed / MEDLINE",        "api": {"available": True,  "python_library": "Biopython (Bio.Entrez)",           "key_required": False}},
        {"id": "embase",        "name": "Embase",                  "api": {"available": False, "fallback": "Manual RIS export"}},
        {"id": "acl_anthology", "name": "ACL Anthology",           "api": {"available": True,  "python_library": "local BibTeX dump",                "key_required": False}},
        {"id": "ieee_xplore",   "name": "IEEE Xplore",             "api": {"available": True,  "python_library": "requests",                        "key_required": True}},
        {"id": "acm_dl",        "name": "ACM Digital Library",     "api": {"available": False, "fallback": "Manual BibTeX export"}}
    ],
    "tier2_preprints": [
        {"id": "arxiv",    "name": "arXiv",    "categories": ["cs.CL", "cs.AI", "cs.IR"], "api": {"available": True, "python_library": "arxiv (pip)",   "key_required": False}},
        {"id": "medrxiv",  "name": "medRxiv",  "api": {"available": True, "base_url": "https://api.biorxiv.org/details/medrxiv/",  "key_required": False}},
        {"id": "biorxiv",  "name": "bioRxiv",  "api": {"available": True, "base_url": "https://api.biorxiv.org/details/biorxiv/",  "key_required": False}}
    ]
}

api_yes = [db["id"] for tier in DATABASES.values() for db in tier if db["api"]["available"]]
api_no  = [db["id"] for tier in DATABASES.values() for db in tier if not db["api"]["available"]]
print("Programmatic:", api_yes)
print("Manual export:", api_no)

---
## 3. Stage 1 Metadata Schema

In [ ]:
@dataclass
class PaperRecord:
    title:     str
    authors:   list[str]
    year:      Optional[int]
    doi:       Optional[str]   # primary dedup key
    abstract:  Optional[str]   # required for title/abstract screening
    bibtex:    Optional[str]   # full BibTeX string
    source_db: str             # which database retrieved from
    url:       Optional[str]  = field(default=None)
    venue:     Optional[str]  = field(default=None)

    def to_dict(self) -> dict:
        return {
            "title":     self.title,
            "authors":   "; ".join(self.authors),
            "year":      self.year,
            "doi":       self.doi,
            "abstract":  self.abstract,
            "bibtex":    self.bibtex,
            "source_db": self.source_db,
            "url":       self.url,
            "venue":     self.venue,
        }

def records_to_df(records: list[PaperRecord]) -> pd.DataFrame:
    return pd.DataFrame([r.to_dict() for r in records])

print("PaperRecord fields:", list(PaperRecord.__dataclass_fields__.keys()))

---
## 4. Query Construction

Builds one query string per database from `VOCABULARY`.
Base structure: `(SNOMED terms) AND (task terms)`.
Date range and field tags are applied per-database syntax.

In [ ]:
SNOMED_VARIANTS = ["SNOMED CT", "SNOMED-CT"]
YEAR_START, YEAR_END = 2016, 2026

def build_queries(task_terms: list[str]) -> dict[str, str]:
    """Return a per-database query dict."""

    # --- PubMed ---
    snomed_pm  = " OR ".join(f'"{v}"[Title/Abstract]' for v in SNOMED_VARIANTS)
    task_pm    = " OR ".join(f'"{t}"[Title/Abstract]' for t in task_terms)
    date_pm    = f'("{YEAR_START}/01/01"[Date - Publication] : "{YEAR_END}/12/31"[Date - Publication])'
    pubmed     = f"({snomed_pm}) AND ({task_pm}) AND {date_pm}"

    # --- arXiv (arxiv Python lib uses plain boolean text) ---
    # Keep concise: SNOMED + broad NLP anchor + top task terms to avoid >200-char limit issues
    top_task   = task_terms[:20]   # first 20 terms; refine after pilot
    snomed_ax  = " OR ".join(f'"{v}"' for v in SNOMED_VARIANTS)
    task_ax    = " OR ".join(f'"{t}"' for t in top_task)
    arxiv_q    = f"({snomed_ax}) AND ({task_ax})"

    # --- IEEE Xplore (querytext parameter, plain boolean) ---
    snomed_ie  = " OR ".join(f'"{v}"' for v in SNOMED_VARIANTS)
    task_ie    = " OR ".join(f'"{t}"' for t in task_terms)
    ieee       = f"({snomed_ie}) AND ({task_ie})"

    # --- medRxiv / bioRxiv (Cold Spring Harbor search: space-separated keywords) ---
    rxiv       = "SNOMED NLP"

    # --- ACL Anthology (local Python string match; no formal query string needed) ---
    acl        = None   # handled directly in retrieval function

    return {
        "pubmed":        pubmed,
        "arxiv":         arxiv_q,
        "ieee_xplore":   ieee,
        "medrxiv":       rxiv,
        "biorxiv":       rxiv,
        "acl_anthology": acl,
    }

QUERIES = build_queries(ALL_TASK_TERMS)

for db, q in QUERIES.items():
    preview = (q[:120] + "...") if q and len(q) > 120 else q
    print(f"[{db}]\n  {preview}\n")

---
## 5. Paper Retrieval

One function per database. Each returns `list[PaperRecord]`.
Set your credentials in the config cell below before running.

In [ ]:
# ── Load credentials from .env ─────────────────────────────────────────────
# Copy .env.example → .env and fill in your values. Never commit .env.
load_dotenv()

CONFIG = {
    "pubmed_email":    os.getenv("PUBMED_EMAIL", ""),
    "pubmed_api_key":  os.getenv("PUBMED_API_KEY"),      # None if unset
    "ieee_api_key":    os.getenv("IEEE_API_KEY"),         # register at developer.ieee.org
    "acl_bib_path":    os.getenv("ACL_BIB_PATH", "anthology+abstracts.bib"),
    "embase_ris_path": os.getenv("EMBASE_RIS_PATH"),
    "acm_bib_path":    os.getenv("ACM_BIB_PATH"),
    "output_dir":      Path("results"),
}

# Sanity check
missing = [k for k in ("pubmed_email",) if not CONFIG[k]]
if missing:
    print(f"WARNING: missing required env vars: {missing}")
else:
    print("Config loaded OK")

In [ ]:
# ── 5a. PubMed ─────────────────────────────────────────────────────────────

def _parse_pubmed_record(rec: dict) -> PaperRecord:
    """Convert a Biopython Medline record dict to PaperRecord."""
    title   = rec.get("TI", "")
    authors = rec.get("AU", [])
    venue   = rec.get("TA") or rec.get("JT")   # abbreviated / full journal title
    abstract= rec.get("AB")
    pmid    = rec.get("PMID", "")

    # Year: try DP (date of publication) field first
    dp   = rec.get("DP", "")
    year = int(dp[:4]) if dp and dp[:4].isdigit() else None

    # DOI: lives in AID field as "10.xxx/xxx [doi]"
    doi = None
    for aid in rec.get("AID", []):
        if "[doi]" in aid:
            doi = aid.replace(" [doi]", "").strip()
            break

    bibtex = (
        f"@article{{pmid{pmid},\n"
        f"  title   = {{{title}}},\n"
        f"  author  = {{{' and '.join(authors)}}},\n"
        f"  journal = {{{venue or ''}}},\n"
        f"  year    = {{{year or ''}}},\n"
        f"  doi     = {{{doi or ''}}},\n"
        f"  pmid    = {{{pmid}}}\n}}"
    )

    return PaperRecord(
        title=title, authors=authors, year=year, doi=doi,
        abstract=abstract, bibtex=bibtex, source_db="pubmed",
        url=f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else None,
        venue=venue,
    )


def retrieve_pubmed(query: str, cfg: dict, max_results: int = 5000) -> list[PaperRecord]:
    Entrez.email   = cfg["pubmed_email"]
    Entrez.api_key = cfg["pubmed_api_key"]   # None is fine
    delay          = 0.11 if cfg["pubmed_api_key"] else 0.34  # ~10 or ~3 req/s

    # Search → get PMIDs
    handle  = Entrez.esearch(db="pubmed", term=query, retmax=max_results, usehistory="y")
    search  = Entrez.read(handle)
    handle.close()
    total   = int(search["Count"])
    print(f"PubMed: {total} hits (fetching up to {max_results})")

    # Fetch records in batches via WebEnv
    records, batch = [], 500
    for start in tqdm(range(0, min(total, max_results), batch), desc="PubMed batches"):
        fh = Entrez.efetch(
            db="pubmed", rettype="medline", retmode="text",
            retstart=start, retmax=batch,
            webenv=search["WebEnv"], query_key=search["QueryKey"]
        )
        for rec in Medline.parse(fh):
            records.append(_parse_pubmed_record(rec))
        fh.close()
        time.sleep(delay)

    return records

In [ ]:
# ── 5b. arXiv ──────────────────────────────────────────────────────────────

def retrieve_arxiv(query: str, max_results: int = 1000) -> list[PaperRecord]:
    client = arxiv.Client(page_size=100, delay_seconds=3)
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    records = []
    for r in tqdm(client.results(search), desc="arXiv", total=max_results):
        year = r.published.year if r.published else None
        if year and not (YEAR_START <= year <= YEAR_END):
            continue

        arxiv_id = r.entry_id.split("/")[-1]
        bibtex = (
            f"@misc{{{arxiv_id},\n"
            f"  title   = {{{r.title}}},\n"
            f"  author  = {{{' and '.join(str(a) for a in r.authors)}}},\n"
            f"  year    = {{{year or ''}}},\n"
            f"  eprint  = {{{arxiv_id}}},\n"
            f"  archivePrefix = {{arXiv}},\n"
            f"  url     = {{{r.entry_id}}}\n}}"
        )

        records.append(PaperRecord(
            title=r.title,
            authors=[str(a) for a in r.authors],
            year=year,
            doi=r.doi,
            abstract=r.summary,
            bibtex=bibtex,
            source_db="arxiv",
            url=r.entry_id,
            venue="arXiv",
        ))

    print(f"arXiv: {len(records)} records after year filter")
    return records

In [ ]:
# ── 5c. medRxiv / bioRxiv (Cold Spring Harbor API) ─────────────────────────
# The CSH API supports keyword search via:
# GET https://api.biorxiv.org/search/{server}/{terms}/{cursor}/json
# Returns up to 100 results per page; cursor is the offset.

def _parse_rxiv_record(rec: dict, server: str) -> PaperRecord:
    doi    = rec.get("doi")
    year_s = rec.get("date", "")[:4]
    year   = int(year_s) if year_s.isdigit() else None
    authors= [a.strip() for a in rec.get("authors", "").split(";") if a.strip()]

    bibtex = (
        f"@article{{{doi.replace('/', '_') if doi else 'unknown'},\n"
        f"  title   = {{{rec.get('title', '')}}},\n"
        f"  author  = {{{' and '.join(authors)}}},\n"
        f"  year    = {{{year or ''}}},\n"
        f"  doi     = {{{doi or ''}}},\n"
        f"  journal = {{{server}}}\n}}"
    )

    return PaperRecord(
        title=rec.get("title", ""),
        authors=authors,
        year=year,
        doi=doi,
        abstract=rec.get("abstract"),
        bibtex=bibtex,
        source_db=server,
        url=f"https://doi.org/{doi}" if doi else None,
        venue=server,
    )


def retrieve_rxiv(query: str, server: str = "medrxiv", max_results: int = 500) -> list[PaperRecord]:
    """server: 'medrxiv' or 'biorxiv'"""
    base    = f"https://api.biorxiv.org/search/{server}/{query}"
    records = []
    cursor  = 0

    with tqdm(desc=server, total=max_results) as pbar:
        while len(records) < max_results:
            resp = requests.get(f"{base}/{cursor}/json", timeout=30)
            resp.raise_for_status()
            data = resp.json()

            messages = data.get("messages", [])
            if not messages or messages[0].get("status") == "no posts found":
                break

            collection = data.get("collection", [])
            if not collection:
                break

            for rec in collection:
                year_s = rec.get("date", "")[:4]
                year   = int(year_s) if year_s.isdigit() else None
                if year and not (YEAR_START <= year <= YEAR_END):
                    continue
                records.append(_parse_rxiv_record(rec, server))

            pbar.update(len(collection))
            cursor += 100
            time.sleep(1)

    print(f"{server}: {len(records)} records after year filter")
    return records

In [ ]:
# ── 5d. IEEE Xplore ────────────────────────────────────────────────────────

def retrieve_ieee(query: str, api_key: str, max_results: int = 1000) -> list[PaperRecord]:
    if not api_key:
        print("IEEE Xplore: no API key set — skipping. Register at developer.ieee.org")
        return []

    url     = "https://ieeexploreapi.ieee.org/api/v1/search/articles"
    records = []
    start   = 1
    page    = 25   # max per request

    with tqdm(desc="IEEE Xplore", total=max_results) as pbar:
        while start <= max_results:
            params = {
                "apikey":       api_key,
                "querytext":    query,
                "start_record": start,
                "max_records":  min(page, max_results - start + 1),
                "start_year":   YEAR_START,
                "end_year":     YEAR_END,
                "format":       "json",
            }
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()

            articles = data.get("articles", [])
            if not articles:
                break

            for a in articles:
                doi     = a.get("doi")
                year    = int(a.get("publication_year", 0)) or None
                authors = [auth["full_name"] for auth in a.get("authors", {}).get("authors", [])]
                title   = a.get("title", "")
                venue   = a.get("publication_title", "")

                bibtex = (
                    f"@article{{{doi.replace('/', '_') if doi else title[:20].replace(' ', '_')},\n"
                    f"  title   = {{{title}}},\n"
                    f"  author  = {{{' and '.join(authors)}}},\n"
                    f"  journal = {{{venue}}},\n"
                    f"  year    = {{{year or ''}}},\n"
                    f"  doi     = {{{doi or ''}}}\n}}"
                )

                records.append(PaperRecord(
                    title=title, authors=authors, year=year, doi=doi,
                    abstract=a.get("abstract"),
                    bibtex=bibtex, source_db="ieee_xplore",
                    url=a.get("html_url"), venue=venue,
                ))

            pbar.update(len(articles))
            if start == 1:
                total = int(data.get("total_records", 0))
                print(f"  IEEE total hits: {total}")
            start += page
            time.sleep(1)

    print(f"IEEE Xplore: {len(records)} records")
    return records

In [ ]:
# ── 5e. ACL Anthology (local BibTeX dump) ─────────────────────────────────
# Download anthology+abstracts.bib.gz from https://aclanthology.org/anthology+abstracts.bib.gz
# Decompress and set acl_bib_path in CONFIG.

def retrieve_acl_anthology(bib_path: str) -> list[PaperRecord]:
    bib_path = Path(bib_path)
    if not bib_path.exists():
        print(f"ACL Anthology: BibTeX dump not found at {bib_path}")
        print("  Download from: https://aclanthology.org/anthology+abstracts.bib.gz")
        return []

    print(f"Loading ACL BibTeX dump from {bib_path} ...")
    with open(bib_path, encoding="utf-8") as f:
        bib_db = bibtexparser.load(f)
    print(f"  {len(bib_db.entries)} total entries")

    snomed_patterns = [s.lower() for s in SNOMED_VARIANTS]
    task_patterns   = [t.lower() for t in ALL_TASK_TERMS]

    records = []
    for entry in tqdm(bib_db.entries, desc="ACL Anthology"):
        text = (entry.get("title", "") + " " + entry.get("abstract", "")).lower()

        if not any(s in text for s in snomed_patterns):
            continue
        if not any(t in text for t in task_patterns):
            continue

        year_s = entry.get("year", "")
        year   = int(year_s) if year_s.isdigit() else None
        if year and not (YEAR_START <= year <= YEAR_END):
            continue

        authors = [a.strip() for a in re.split(r" and ", entry.get("author", ""), flags=re.IGNORECASE)]
        doi     = entry.get("doi")
        venue   = entry.get("booktitle") or entry.get("journal")

        db = bibtexparser.bibdatabase.BibDatabase()
        db.entries = [entry]
        bibtex = bibtexparser.dumps(db)

        records.append(PaperRecord(
            title=entry.get("title", ""),
            authors=authors, year=year, doi=doi,
            abstract=entry.get("abstract"),
            bibtex=bibtex, source_db="acl_anthology",
            url=entry.get("url"), venue=venue,
        ))

    print(f"ACL Anthology: {len(records)} matching records")
    return records

In [ ]:
# ── 5f. Manual export loader (Embase RIS, ACM BibTeX) ─────────────────────

def _parse_ris_entry(block: str, source_db: str) -> Optional[PaperRecord]:
    """Parse a single RIS entry block into a PaperRecord."""
    fields: dict[str, list[str]] = {}
    for line in block.splitlines():
        if len(line) >= 6 and line[2:4] == "  ":
            tag, val = line[:2], line[6:].strip()
            fields.setdefault(tag, []).append(val)

    title   = " ".join(fields.get("TI", fields.get("T1", [""])))
    authors = fields.get("AU", fields.get("A1", []))
    year_v  = " ".join(fields.get("PY", fields.get("Y1", [""])))[:4]
    year    = int(year_v) if year_v.isdigit() else None
    doi     = " ".join(fields.get("DO", [None] )) if "DO" in fields else None
    abstract= " ".join(fields.get("AB", []))
    venue   = " ".join(fields.get("JO", fields.get("T2", [])))

    if not title:
        return None

    return PaperRecord(
        title=title, authors=authors, year=year, doi=doi or None,
        abstract=abstract or None, bibtex=None,
        source_db=source_db, venue=venue or None,
    )


def load_ris_export(filepath: str, source_db: str) -> list[PaperRecord]:
    """Load a manually exported RIS file (e.g. from Embase)."""
    if not filepath or not Path(filepath).exists():
        print(f"{source_db}: RIS file not found at {filepath} — skipping")
        return []

    text   = Path(filepath).read_text(encoding="utf-8", errors="replace")
    blocks = re.split(r"\nER\s*-?\s*\n", text)
    records = [r for b in blocks if (r := _parse_ris_entry(b, source_db)) is not None]
    print(f"{source_db}: {len(records)} records loaded from RIS")
    return records


def load_bibtex_export(filepath: str, source_db: str) -> list[PaperRecord]:
    """Load a manually exported BibTeX file (e.g. from ACM DL)."""
    if not filepath or not Path(filepath).exists():
        print(f"{source_db}: BibTeX file not found at {filepath} — skipping")
        return []

    with open(filepath, encoding="utf-8") as f:
        bib_db = bibtexparser.load(f)

    records = []
    for entry in bib_db.entries:
        year_s = entry.get("year", "")
        year   = int(year_s) if year_s.isdigit() else None
        authors = [a.strip() for a in re.split(r" and ", entry.get("author", ""), flags=re.IGNORECASE)]
        db2 = bibtexparser.bibdatabase.BibDatabase()
        db2.entries = [entry]

        records.append(PaperRecord(
            title=entry.get("title", ""),
            authors=authors, year=year, doi=entry.get("doi"),
            abstract=entry.get("abstract"),
            bibtex=bibtexparser.dumps(db2),
            source_db=source_db,
            url=entry.get("url"),
            venue=entry.get("journal") or entry.get("booktitle"),
        ))

    print(f"{source_db}: {len(records)} records loaded from BibTeX")
    return records

In [ ]:
# ── 5g. Run all retrievals ─────────────────────────────────────────────────

def run_retrieval(cfg: dict, queries: dict) -> list[PaperRecord]:
    all_records: list[PaperRecord] = []

    # Programmatic sources
    all_records += retrieve_pubmed(queries["pubmed"], cfg)
    all_records += retrieve_arxiv(queries["arxiv"])
    all_records += retrieve_rxiv(queries["medrxiv"], server="medrxiv")
    all_records += retrieve_rxiv(queries["biorxiv"], server="biorxiv")
    all_records += retrieve_ieee(queries["ieee_xplore"], cfg["ieee_api_key"])
    all_records += retrieve_acl_anthology(cfg["acl_bib_path"])

    # Manual exports
    all_records += load_ris_export(cfg["embase_ris_path"],  source_db="embase")
    all_records += load_bibtex_export(cfg["acm_bib_path"],  source_db="acm_dl")

    print(f"\nTotal raw records: {len(all_records)}")

    # Summary by source
    df = records_to_df(all_records)
    print(df.groupby("source_db").size().rename("count").to_string())
    return all_records


def save_records(records: list[PaperRecord], cfg: dict) -> Path:
    cfg["output_dir"].mkdir(exist_ok=True)
    out = cfg["output_dir"] / "raw_retrieval.csv"
    records_to_df(records).to_csv(out, index=False)
    print(f"Saved {len(records)} records to {out}")
    return out


# ── Uncomment to run ───────────────────────────────────────────────────────
# CONFIG["pubmed_email"] = "your@email.com"   # set before running
# raw_records = run_retrieval(CONFIG, QUERIES)
# save_records(raw_records, CONFIG)

---
## 6. Deduplication

Input: raw records from `run_retrieval()` (or reloaded from `results/raw_retrieval.csv`).

**Two-pass strategy:**
1. **Pass 1 — DOI exact match** (normalized: lowercase, strip `https://doi.org/` prefix). Handles the majority of cross-database duplicates.
2. **Pass 2 — Fuzzy title match** (token-sort ratio ≥ 90 via `thefuzz`). Catches records with missing or malformed DOIs. O(n²) but fine for expected corpus size (~500–2 000 records).

For each duplicate group the **first-seen record is kept** (order: PubMed → arXiv → IEEE → ACL → preprints → manual exports, reflecting descending metadata reliability). The `source_dbs` column accumulates all databases that returned that record.

# ── 6a. Normalisation helpers ──────────────────────────────────────────────

_DOI_PREFIX = re.compile(r"^https?://(?:dx\.)?doi\.org/", re.IGNORECASE)

def _norm_doi(doi: Optional[str]) -> Optional[str]:
    if not doi:
        return None
    return _DOI_PREFIX.sub("", doi).lower().strip()

def _norm_title(title: str) -> str:
    t = title.lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


# ── 6b. Source priority (lower index = kept over later sources) ────────────

_SOURCE_ORDER = [
    "pubmed", "embase", "acl_anthology", "ieee_xplore", "acm_dl",
    "arxiv", "medrxiv", "biorxiv",
]

def _source_rank(source_db: str) -> int:
    for i, s in enumerate(_SOURCE_ORDER):
        if s in source_db:
            return i
    return len(_SOURCE_ORDER)


# ── 6c. Deduplication ─────────────────────────────────────────────────────

def deduplicate(
    records: list[PaperRecord],
    fuzzy_threshold: int = 90,
) -> pd.DataFrame:
    """
    Two-pass deduplication.

    Returns a deduplicated DataFrame with a 'source_dbs' column (semicolon-
    separated list of every database that returned the record).
    """
    records_sorted = sorted(records, key=lambda r: _source_rank(r.source_db))
    df = records_to_df(records_sorted).copy()

    df["_doi_norm"]   = df["doi"].apply(_norm_doi)
    df["_title_norm"] = df["title"].apply(_norm_title)
    df["_keep"]       = True
    df["source_dbs"]  = df["source_db"]

    doi_dups = 0
    title_dups = 0

    # Pass 1: DOI exact match
    doi_seen: dict[str, int] = {}
    for i, row in df.iterrows():
        d = row["_doi_norm"]
        if not d:
            continue
        if d in doi_seen:
            df.at[doi_seen[d], "source_dbs"] += f"; {row['source_db']}"
            df.at[i, "_keep"] = False
            doi_dups += 1
        else:
            doi_seen[d] = i

    # Pass 2: fuzzy title match on pass-1 survivors
    surviving_idx = [i for i in df.index if df.at[i, "_keep"]]
    titles        = [df.at[i, "_title_norm"] for i in surviving_idx]

    for pos in range(len(surviving_idx)):
        i = surviving_idx[pos]
        if not df.at[i, "_keep"]:
            continue
        for pos2 in range(pos + 1, len(surviving_idx)):
            j = surviving_idx[pos2]
            if not df.at[j, "_keep"]:
                continue
            if fuzz.token_sort_ratio(titles[pos], titles[pos2]) >= fuzzy_threshold:
                df.at[i, "source_dbs"] += f"; {df.at[j, 'source_db']}"
                df.at[j, "_keep"] = False
                title_dups += 1

    deduped = (
        df[df["_keep"]]
        .drop(columns=["_doi_norm", "_title_norm", "_keep", "source_db"])
        .reset_index(drop=True)
    )

    print(f"Raw records           : {len(df)}")
    print(f"  Pass 1 (DOI exact)  : -{doi_dups}")
    print(f"  Pass 2 (fuzzy≥{fuzzy_threshold}%) : -{title_dups}")
    print(f"Unique records        : {len(deduped)}")
    print()
    print(deduped["source_dbs"].str.split("; ").explode().value_counts().to_string())
    return deduped


def save_deduped(deduped: pd.DataFrame, cfg: dict) -> Path:
    cfg["output_dir"].mkdir(exist_ok=True)
    out = cfg["output_dir"] / "deduped.csv"
    deduped.to_csv(out, index=False)
    print(f"Saved {len(deduped)} deduplicated records to {out}")
    return out

In [ ]:
# ── Run deduplication ─────────────────────────────────────────────────────
# Option A: in-memory from Section 5
# deduped = deduplicate(raw_records)

# Option B: reload from saved CSV after kernel restart
# raw_df = pd.read_csv(CONFIG["output_dir"] / "raw_retrieval.csv")
# raw_records_reload = [
#     PaperRecord(
#         title=r["title"], authors=str(r["authors"]).split("; "),
#         year=int(r["year"]) if pd.notna(r["year"]) else None,
#         doi=r["doi"] if pd.notna(r["doi"]) else None,
#         abstract=r["abstract"] if pd.notna(r["abstract"]) else None,
#         bibtex=r["bibtex"] if pd.notna(r["bibtex"]) else None,
#         source_db=r["source_db"],
#         url=r["url"] if pd.notna(r["url"]) else None,
#         venue=r["venue"] if pd.notna(r["venue"]) else None,
#     )
#     for _, r in raw_df.iterrows()
# ]
# deduped = deduplicate(raw_records_reload)

# save_deduped(deduped, CONFIG)

---
## 7. Screening

Input: deduplicated DataFrame from Section 6.

**Automated pre-filter** assigns one of three statuses to each record based on title + abstract signals:

| Status | Meaning |
|---|---|
| `auto_exclude` | Fails a hard criterion (year out of range, or no SNOMED CT mention in title/abstract) |
| `auto_include` | Strong positive signal on all three axes: SNOMED explicit + DL method + clinical domain |
| `needs_review` | SNOMED present but DL or clinical signal is ambiguous → manual title/abstract check |

All `auto_include` and `needs_review` records proceed to manual title/abstract screening (PRISMA step 2), then full-text eligibility review (PRISMA step 3). Output CSVs seed the review spreadsheet.

In [ ]:
# ── 7a. Signal term lists ──────────────────────────────────────────────────

_SNOMED_EXPLICIT = ["snomed ct", "snomed-ct"]

_DL_SIGNALS = [
    "deep learning", "neural network", "transformer", "attention mechanism",
    "bert", "biobert", "clinicalbert", "pubmedbert", "roberta",
    "gpt", "llm", "large language model", "chatgpt",
    "llama", "mistral", "falcon", "gemini",
    "lstm", "rnn", "gru", "recurrent neural",
    "seq2seq", "encoder-decoder", "fine-tun", "pre-train", "pretrain",
    "embedding", "word2vec", "fasttext",
    "agentic", "retrieval augmented", "rag",
]

_CLINICAL_SIGNALS = [
    "clinical", "patient", "ehr", "electronic health record",
    "hospital", "physician", "clinician", "nurse",
    "medication", "drug", "diagnosis", "icd",
    "discharge summary", "radiology report", "pathology report",
    "medical record", "health informatics", "biomedical",
    "clinical note", "clinical text", "clinical nlp",
]


# ── 7b. Screener ──────────────────────────────────────────────────────────

def screen_records(deduped: pd.DataFrame) -> pd.DataFrame:
    df = deduped.copy()
    statuses, reasons = [], []

    for _, row in df.iterrows():
        text = f"{row.get('title', '') or ''} {row.get('abstract', '') or ''}".lower()

        year = row.get("year")
        if year and not (YEAR_START <= int(year) <= YEAR_END):
            statuses.append("auto_exclude")
            reasons.append(f"year {year} outside {YEAR_START}–{YEAR_END}")
            continue

        has_snomed = any(s in text for s in _SNOMED_EXPLICIT)
        if not has_snomed:
            statuses.append("auto_exclude")
            reasons.append("no SNOMED CT / SNOMED-CT mention in title or abstract")
            continue

        has_dl       = any(d in text for d in _DL_SIGNALS)
        has_clinical = any(c in text for c in _CLINICAL_SIGNALS)

        if has_dl and has_clinical:
            statuses.append("auto_include")
            reasons.append("SNOMED + DL method + clinical domain signals all present")
        elif has_dl:
            statuses.append("needs_review")
            reasons.append("SNOMED + DL signal present; clinical domain signal absent")
        elif has_clinical:
            statuses.append("needs_review")
            reasons.append("SNOMED + clinical signal present; DL method signal absent")
        else:
            statuses.append("needs_review")
            reasons.append("SNOMED present; no DL or clinical signal — manual check required")

    df["screen_status"] = statuses
    df["screen_reason"]  = reasons

    counts = df["screen_status"].value_counts()
    print("Screening results:")
    for status, count in counts.items():
        print(f"  {status:15s}: {count:4d}")
    print(f"\n  → {counts.get('auto_include', 0) + counts.get('needs_review', 0)} records forwarded to manual review")
    return df


def save_screened(screened: pd.DataFrame, cfg: dict) -> Path:
    cfg["output_dir"].mkdir(exist_ok=True)
    out = cfg["output_dir"] / "screened.csv"
    screened.to_csv(out, index=False)

    for_review = (
        screened[screened["screen_status"] != "auto_exclude"]
        .sort_values(["screen_status", "year"], ascending=[True, False])
        .reset_index(drop=True)
    )
    review_out = cfg["output_dir"] / "for_manual_review.csv"
    for_review.to_csv(review_out, index=False)

    print(f"Saved {len(screened)} screened records → {out}")
    print(f"Saved {len(for_review)} records for manual review → {review_out}")
    return out

# ── Run screening ─────────────────────────────────────────────────────────
# screened = screen_records(deduped)
# save_screened(screened, CONFIG)

# Preview
# screened[screened["screen_status"] == "auto_include"][["title", "year", "source_dbs", "venue"]].head(20)